# 22_forward_model_inverted_pendulum_dagger — Faz 1 (DAgger + Forward Model + MPC)
**Model-Based RL + DAgger:**
- Ortam: **InvertedPendulum-v4** (MuJoCo, continuous)
- Başlangıç veri: Uzun PPO eğitimi + gürültülü politika
- İleri model (FM): MLP (delta-state), **çok adımlı (multi-step)** kayıp fonksiyonu ile eğitilmiş.
- Planlayıcı: **Random Shooting MPC** (Gaussian)
- **DAgger döngüsü**: MPC ile rollout → gerçek ortamdan (s,a,s′) topla (öğretmen PPO karışımı ile) → veriye ekle → FM’i kısa yeniden eğit → yinele
- Karşılaştırma: **Gerçek vs Model** zaman-serileri + RMSE; **Return — MPC vs PPO**
- Pygame **render demoları** (MPC & PPO)

In [ ]:
# Gerekli kütüphaneleri yüklemek için bu satırın başındaki yorumu kaldırabilirsiniz.
# %pip install -U gymnasium mujoco stable-baselines3 tensorboard pandas matplotlib tqdm pygame

## 1) Kütüphaneler & Bootstrap (sabit düzen)

In [ ]:
import os, sys, time, uuid, json, random, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.logger import configure
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.utils import set_random_seed
from tensorboard.backend.event_processing import event_accumulator as ea

class RLBootstrap:
    def __init__(self, phase, notebook, step, env_id, seed=42):
        self.phase, self.notebook, self.step, self.env_id = phase, notebook, step, env_id
        self.seed = seed
        set_random_seed(seed); np.random.seed(seed); random.seed(seed)
        self.run_id   = time.strftime('%Y%m%d-%H%M%S') + f"-{uuid.uuid4().hex[:6]}"
        self.root_dir = os.path.join('runs', self.phase, self.notebook, self.run_id)
        self.ckpt_dir = os.path.join(self.root_dir, 'checkpoints')
        self.mon_dir  = os.path.join(self.root_dir, 'monitor')
        os.makedirs(self.ckpt_dir, exist_ok=True); os.makedirs(self.mon_dir, exist_ok=True)
        self._write_meta()
    def algo_dir(self, algo_name):
        safe_env = self.env_id.replace('-', '_')
        d = os.path.join(self.root_dir, f"{algo_name}_{safe_env}")
        os.makedirs(d, exist_ok=True)
        return d
    def make_env(self, render=False, monitor=True):
        env = gym.make(self.env_id, render_mode='human') if render else gym.make(self.env_id)
        if monitor:
            md = os.path.join(self.mon_dir, self.env_id.replace('-', '_'))
            os.makedirs(md, exist_ok=True)
            env = Monitor(env, md)
        return env
    def _write_meta(self):
        m = dict(phase=self.phase, notebook=self.notebook, step=self.step, env=self.env_id,
                 seed=self.seed, run_id=self.run_id, root_dir=self.root_dir,
                 ckpt_dir=self.ckpt_dir, monitor_dir=self.mon_dir)
        os.makedirs(self.root_dir, exist_ok=True)
        with open(os.path.join(self.root_dir, 'meta.json'), 'w') as f:
            json.dump(m, f, indent=2)

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Kullanılan Cihaz:', device)

## 2) Ayarlar / Klasörler

In [ ]:
PHASE    = 'Phase 1'
NOTEBOOK = '22_forward_model_inverted_pendulum_dagger'
STEP     = 'ForwardModel+MPC+DAgger_v2'
ENV_ID   = 'InvertedPendulum-v4'
SEED     = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

boot = RLBootstrap(PHASE, NOTEBOOK, STEP, ENV_ID, seed=SEED)
FM_LOG    = boot.algo_dir('ForwardModel')
MPC_LOG   = boot.algo_dir('MPC_RS')
DATA_LOG  = boot.algo_dir('DataGen')
BASE_LOG  = boot.algo_dir('Baseline_PPO')
DAGGER_LOG= boot.algo_dir('DAgger')

CKPT_FM_DIR    = os.path.join(boot.ckpt_dir, 'ForwardModel')
CKPT_MPC_DIR   = os.path.join(boot.ckpt_dir, 'MPC')
CKPT_BASE_DIR  = os.path.join(boot.ckpt_dir, 'Baseline_PPO')
CKPT_DAGGER_DIR= os.path.join(boot.ckpt_dir, 'DAgger')
for d in [CKPT_FM_DIR, CKPT_MPC_DIR, CKPT_BASE_DIR, CKPT_DAGGER_DIR]: os.makedirs(d, exist_ok=True)

print('Kök Dizin:', boot.root_dir)

## 3) Dinamik Model (MLP, delta-state) ve MPC (Gaussian Random Shooting)

In [ ]:
class MLPDynamics(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=(256, 256), layernorm=True):
        super().__init__()
        layers=[]; last=obs_dim+act_dim
        for h in hidden:
            layers += [nn.Linear(last, h), nn.ReLU()]
            if layernorm: layers += [nn.LayerNorm(h)]
            last=h
        layers += [nn.Linear(last, obs_dim)]
        self.net = nn.Sequential(*layers)
    def forward(self, s, a):
        return self.net(torch.cat([s,a], dim=-1))
    def step(self, s, a):
        # Bu fonksiyon sadece çıkarım (inference) ve MPC planlaması için kullanılır, eğitimde değil.
        with torch.no_grad():
            return s + self.forward(s,a)  # delta-state

# Ortamın ödül fonksiyonunu PyTorch tensörleriyle yeniden tanımlıyoruz
def invpend_reward_torch(obs, act):
    # Bu fonksiyon, orijinal InvertedPendulum-v4 ödül fonksiyonunun bir benzeridir.
    # Gerçek ödül `1.0`'dır ve sonlanma durumunda `0.0` olur.
    return torch.ones(obs.shape[0], device=obs.device)

class RandomShootingMPC:
    def __init__(self, model, act_low, act_high, horizon=15, n_samples=4096, device=device, gauss_std=0.8):
        self.model=model; self.low=float(act_low); self.high=float(act_high)
        self.H=horizon; self.N=n_samples; self.dev=device; self.gauss_std=gauss_std
        self.act_dim = 1 # InvertedPendulum için aksiyon boyutu
        
    def act(self, s_now):
        s0 = torch.as_tensor(s_now, dtype=torch.float32, device=self.dev).unsqueeze(0) if not torch.is_tensor(s_now) else (s_now.to(self.dev).unsqueeze(0) if s_now.ndim==1 else s_now.to(self.dev))
        s0 = s0.repeat(self.N, 1)
        
        # Rastgele aksiyon dizileri oluştur
        acts = (torch.randn(self.N, self.H, self.act_dim, device=self.dev) * self.gauss_std).clamp(self.low, self.high)
        
        s = s0
        total_rewards = torch.zeros(self.N, device=self.dev)
        
        # Ufuk boyunca her bir aksiyon dizisini simüle et
        with torch.no_grad():
            for t in range(self.H):
                a_t = acts[:,t,:]
                total_rewards += invpend_reward_torch(s, a_t)
                s = self.model.step(s, a_t)
                
        # En yüksek toplam ödüle sahip aksiyon dizisini bul
        best_idx = torch.argmax(total_rewards)
        # Bu dizinin ilk aksiyonunu seç
        best_action = acts[best_idx, 0, :]
        
        return best_action.clamp(self.low, self.high).cpu().numpy()

## 4) Veri Toplama — Uzun PPO + Gürültülü Politika (kaliteli başlangıç datası)

In [ ]:
# İYİLEŞTİRME: Başlangıç verisini toplamak için daha uzun bir PPO eğitimi yapıyoruz.
PPO_TRAIN_STEPS = 200_000
NOISY_STEPS     = 100_000
NOISE_STD       = 0.3 # Daha iyi bir politikamız olduğu için gürültüyü azalttık

train_env = boot.make_env(monitor=True)
eval_env = boot.make_env(monitor=False)

print("Başlangıç PPO politikası eğitiliyor...")
ppo = PPO('MlpPolicy', train_env, device=device, seed=SEED, verbose=0)
ppo.set_logger(configure(DATA_LOG, ['stdout','csv','tensorboard']))
ppo.learn(total_timesteps=PPO_TRAIN_STEPS, progress_bar=True)
ppo.save(os.path.join(CKPT_BASE_DIR, "ppo_initial_teacher"))

mean_rew, std_rew = evaluate_policy(ppo, eval_env, n_eval_episodes=10)
print(f"Eğitilmiş PPO Öğretmen: Ortalama Ödül = {mean_rew:.2f} +/- {std_rew:.2f}")

train_env.close()
eval_env.close()

print(f"\n{NOISY_STEPS} adımlık gürültülü veri toplanıyor...")
env = boot.make_env(monitor=True)
obs_list, act_list, next_list = [], [], []
obs, info = env.reset(seed=SEED+123)
for _ in tqdm(range(NOISY_STEPS), desc='Gürültülü PPO ile veri toplanıyor'):
    a, _ = ppo.predict(obs, deterministic=False)
    a = a + NOISE_STD * np.random.randn(*a.shape)
    a = np.clip(a, env.action_space.low, env.action_space.high)
    nobs, r, done, trunc, _ = env.step(a)
    obs_list.append(obs); act_list.append(a); next_list.append(nobs)
    obs = nobs
    if done or trunc:
        obs, info = env.reset()
env.close()

OBS  = np.array(obs_list,  dtype=np.float32)
ACT  = np.array(act_list,  dtype=np.float32)
NEXT = np.array(next_list, dtype=np.float32)
print('\nBaşlangıç veri seti oluşturuldu:', OBS.shape, ACT.shape, NEXT.shape)

## 5) Dataset / DataLoader (delta-state hedefi)

In [ ]:
class FMDataset(Dataset):
    def __init__(self, obs, act, next_obs):
        self.obs, self.act, self.next = obs, act, next_obs
    def __len__(self): return len(self.obs)
    def __getitem__(self, i):
        s = self.obs[i]; a = self.act[i]; ns = self.next[i]
        y = ns - s
        return s, a, y, ns

def make_loaders(OBS, ACT, NEXT, batch=1024):
    N = len(OBS); idx = np.arange(N); np.random.shuffle(idx)
    train_idx = idx[: int(0.9*N)]; val_idx = idx[int(0.9*N):]
    train_ds = FMDataset(OBS[train_idx], ACT[train_idx], NEXT[train_idx])
    val_ds   = FMDataset(OBS[val_idx],   ACT[val_idx],   NEXT[val_idx])
    train_loader = DataLoader(train_ds, batch_size=batch, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch, shuffle=False, drop_last=False)
    return train_loader, val_loader

train_loader, val_loader = make_loaders(OBS, ACT, NEXT)
print('Eğitim/Doğrulama Veri Boyutları:', len(train_loader.dataset), len(val_loader.dataset))

## 6) FM Eğitimi + k-adım MSE (başlangıç modeli)

In [ ]:
obs_dim, act_dim = OBS.shape[1], ACT.shape[1]
# İYİLEŞTİRME: Aşırı uyumu (overfitting) önlemek için daha küçük bir model mimarisi kullanıyoruz.
fm = MLPDynamics(obs_dim, act_dim, hidden=(256, 256), layernorm=True).to(device)
opt = optim.Adam(fm.parameters(), lr=3e-4)
writer_fm = SummaryWriter(FM_LOG)
ckpt_fm = os.path.join(CKPT_FM_DIR, 'forward_model.pt')


def multi_step_mse_seq(model, OBS, ACT, NEXT, k=5, max_traj=60):
    errs=[]; T = len(OBS); stride = max(1, (T//max_traj))
    model.eval() # Değerlendirme modu
    with torch.no_grad():
        for start in range(0, T-k-1, stride):
            s = torch.tensor(OBS[start], device=device).unsqueeze(0)
            for t in range(k):
                a_t = torch.tensor(ACT[start+t], device=device).unsqueeze(0)
                s = model.step(s, a_t)
            # DÜZELTME: Doğru indeksleme (k adım sonrası)
            true_k = torch.tensor(NEXT[start+k-1], device=device).unsqueeze(0)
            errs.append(torch.mean((s-true_k)**2).item())
    return float(np.mean(errs)) if errs else 0.0

# DÜZELTME: Eğitim sırasında gradyan akışını sağlamak için yeni rollout fonksiyonu
def sequence_rollout(model, s_start, a_seq):
    s_preds = []
    s = s_start
    for t in range(a_seq.shape[1]):
        # model.step() `no_grad` içerdiği için eğitimde doğrudan forward çağrısı yapıyoruz
        delta_s = model(s, a_seq[:, t, :])
        s = s + delta_s
        s_preds.append(s)
    return torch.stack(s_preds, dim=1)

def train_fm(model, train_loader, val_loader, epochs=100, tag='init', k_steps=[3, 5, 10]):
    best=float('inf'); no_imp=0; patience=10; gs=0
    # İYİLEŞTİRME: Çok adımlı kayıp için ağırlıklar
    loss_weights = {'1_step': 1.0, '3_step': 0.5, '5_step': 0.25, '10_step': 0.1}
    
    for ep in range(1, epochs+1):
        model.train()
        for s_batch, a_batch, y_batch, ns_batch in train_loader:
            s=s_batch.to(device); a=a_batch.to(device); y=y_batch.to(device); ns=ns_batch.to(device)
            
            # --- 1. Adım Kaybı ---
            pred_1_step = model(s,a)
            loss_1_step = nn.MSELoss()(pred_1_step, y)
            
            # --- K-Adım Kaybı --- 
            total_k_step_loss = torch.tensor(0.0, device=device)
            max_k = max(k_steps)
            
            if a_batch.shape[0] > max_k:
                # DÜZELTME: Boyut uyuşmazlığı hatasını gidermek için dilimleme
                s_start_k = s_batch[:-max_k]                               
                a_seq_k   = a_batch.unfold(0, max_k, 1).permute(0, 2, 1)[:-1]
                ns_seq_k  = ns_batch.unfold(0, max_k, 1).permute(0, 2, 1)[:-1]
                
                pred_seq_k = sequence_rollout(model, s_start_k, a_seq_k)
                
                for k in k_steps:
                    loss_k = nn.MSELoss()(pred_seq_k[:, k-1, :], ns_seq_k[:, k-1, :])
                    total_k_step_loss += loss_weights.get(f'{k}_step', 0.1) * loss_k

            # --- Toplam Kayıp --- 
            total_loss = loss_weights['1_step'] * loss_1_step + total_k_step_loss
            
            opt.zero_grad(); total_loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
            writer_fm.add_scalar(f'{tag}/train_total_loss', total_loss.item(), gs); gs += 1
            
        # --- Doğrulama --- 
        model.eval(); val=0.0; n=0
        with torch.no_grad():
            for s,a,y,_ in val_loader:
                s=s.to(device); a=a.to(device); y=y.to(device)
                val += nn.MSELoss()(model(s,a), y).item(); n+=1
        val/=max(1,n)
        
        k5  = multi_step_mse_seq(model, OBS, ACT, NEXT, k=5)
        k10 = multi_step_mse_seq(model, OBS, ACT, NEXT, k=10)
        writer_fm.add_scalar(f'{tag}/val_1step_mse',  val, ep)
        writer_fm.add_scalar(f'{tag}/val_k5_mse',     k5,  ep)
        writer_fm.add_scalar(f'{tag}/val_k10_mse',    k10, ep)
        print(f"[{tag}][EP {ep:03d}] val={val:.6f} k5={k5:.6f} k10={k10:.6f}")
        
        # Erken durdurma
        if val < best - 1e-7:
            best=val; torch.save(model.state_dict(), ckpt_fm); no_imp=0
        else:
            no_imp+=1
        if no_imp>=patience:
            print('[Erken Durdurma]'); break
            
    return best

print("Başlangıç Dinamik Modeli eğitiliyor...")
_ = train_fm(fm, train_loader, val_loader, epochs=100, tag='init')
writer_fm.close()
print('Dinamik Model kaydedildi ->', ckpt_fm)

## 7) MPC Değerlendirme (başlangıç modeli) + PPO baseline

In [ ]:
fm2 = MLPDynamics(obs_dim, act_dim, hidden=(256, 256), layernorm=True).to(device)
fm2.load_state_dict(torch.load(ckpt_fm, map_location=device))

tmp = gym.make(ENV_ID)
ACT_LOW, ACT_HIGH = float(tmp.action_space.low[0]), float(tmp.action_space.high[0])
tmp.close()
# İYİLEŞTİRME: MPC parametreleri güncellendi
planner = RandomShootingMPC(fm2, ACT_LOW, ACT_HIGH, horizon=15, n_samples=4096, device=device, gauss_std=0.8)

def run_mpc(planner, episodes=10, max_steps=1000, gamma=0.99):
    env = boot.make_env(monitor=False); rets=[]; discs=[]
    for ep in range(episodes):
        obs,_ = env.reset(seed=SEED+ep+100); ret=0.0; disc=0.0; g=1.0
        for t in range(max_steps):
            a = planner.act(obs)
            obs,r,done,trunc,_ = env.step(a)
            ret += r; disc += g*r; g*=gamma
            if done or trunc: break
        rets.append(ret); discs.append(disc)
    env.close(); return float(np.mean(rets)), float(np.std(rets)), float(np.mean(discs))

mpc_mean, mpc_std, mpc_disc = run_mpc(planner)
print(f"\n[MPC-init] Ortalama Ödül = {mpc_mean:.1f} ± {mpc_std:.1f} | İndirgenmiş Ödül = {mpc_disc:.1f}")
w = SummaryWriter(MPC_LOG); w.add_scalar('eval/return_mean', mpc_mean, 0); w.add_scalar('eval/return_std', mpc_std, 0); w.add_scalar('eval/return_disc', mpc_disc, 0); w.close()

print("\nReferans PPO modeli yükleniyor ve değerlendiriliyor...")
ppo_baseline = PPO.load(os.path.join(CKPT_BASE_DIR, "ppo_initial_teacher"), device=device)
eval_env = boot.make_env(monitor=False)
mean_r, std_r = evaluate_policy(ppo_baseline, eval_env, n_eval_episodes=10, deterministic=True)
print(f"[PPO baseline] Ortalama Ödül = {mean_r:.1f} ± {std_r:.1f}")
eval_env.close()

## 8) **DAgger**: MPC roll-out + Öğretmen PPO karışımı (β) → veri agregasyonu → kısa FM yeniden eğitimleri

In [ ]:
# İYİLEŞTİRME: DAgger hiperparametreleri güncellendi
DAGGER_K           = 15
DAGGER_EPISODES    = 10
DAGGER_STEPS       = 1000
BETA_START         = 0.9
BETA_END           = 0.05
NOISE_DAGGER_STD   = 0.2
RETRAIN_EPOCHS     = 40

def beta_schedule(k, K):
    # Lineer azalan beta
    return BETA_START + (BETA_END - BETA_START) * (k / max(1,K-1))

def dagger_collect_and_retrain(OBS_agg, ACT_agg, NEXT_agg):
    global fm, planner # Global model ve planlayıcıyı güncelleyeceğiz
    
    # En son eğitilmiş modeli yükle
    fm.load_state_dict(torch.load(ckpt_fm, map_location=device))
    planner.model = fm
    
    dagger_writer = SummaryWriter(DAGGER_LOG)
    
    for k in range(DAGGER_K):
        beta = beta_schedule(k, DAGGER_K)
        print(f"\n[DAgger] Yineleme {k+1}/{DAGGER_K} | beta={beta:.2f}")
        
        # MPC ile veri topla
        env = boot.make_env(monitor=False)
        new_obs, new_act, new_next = [], [], []
        for ep in tqdm(range(DAGGER_EPISODES), desc=f'DAgger {k+1} veri toplama'):
            s,_ = env.reset(seed=SEED + 500 + 10*k + ep)
            for t in range(DAGGER_STEPS):
                a_mpc = planner.act(s)
                a_tea, _ = ppo_baseline.predict(s, deterministic=True)
                
                a_mix = (1.0 - beta) * a_mpc + beta * a_tea
                a_mix = a_mix + NOISE_DAGGER_STD * np.random.randn(*a_mix.shape)
                a_mix = np.clip(a_mix, env.action_space.low, env.action_space.high)
                
                s2, r, done, trunc, _ = env.step(a_mix)
                new_obs.append(s); new_act.append(a_mix); new_next.append(s2)
                s = s2
                if done or trunc: break
        env.close()
        
        # Veri setini birleştir (agregasyon)
        OBS_agg = np.concatenate([OBS_agg,  np.array(new_obs,  dtype=np.float32)], axis=0)
        ACT_agg = np.concatenate([ACT_agg,  np.array(new_act,  dtype=np.float32)], axis=0)
        NEXT_agg= np.concatenate([NEXT_agg, np.array(new_next, dtype=np.float32)], axis=0)
        print(f'Veri seti boyutu: {OBS_agg.shape[0]}')
        
        # Modeli yeni veriyle yeniden eğit
        tr_loader, va_loader = make_loaders(OBS_agg, ACT_agg, NEXT_agg)
        train_fm(fm, tr_loader, va_loader, epochs=RETRAIN_EPOCHS, tag=f'dagger{k+1}')
        
        # Planlayıcıyı güncellenmiş modelle yenile
        planner.model = fm
        
        # Bu iterasyonun MPC performansını değerlendir ve logla
        mpc_mean, mpc_std, mpc_disc = run_mpc(planner, episodes=5) # Değerlendirmeyi hızlandırmak için 5 bölüm
        print(f"[DAgger {k+1}] MPC Performansı: {mpc_mean:.1f} ± {mpc_std:.1f}")
        dagger_writer.add_scalar('eval/return_mean', mpc_mean, k + 1)
        dagger_writer.add_scalar('eval/return_std', mpc_std, k + 1)
        dagger_writer.add_scalar('eval/return_disc', mpc_disc, k + 1)
        dagger_writer.add_scalar('sizes/obs',  len(OBS_agg),  k + 1)
        
    dagger_writer.close()
    return OBS_agg, ACT_agg, NEXT_agg

OBS, ACT, NEXT = dagger_collect_and_retrain(OBS, ACT, NEXT)
print('\n[DAgger] tamamlandı. Toplam veri boyutu:', OBS.shape[0])

## 9) FM vs Gerçek — Zaman Serisi Karşılaştırmaları

In [ ]:
labels = ['x', 'x_dot', 'theta', 'theta_dot']

def rollout_compare(model, policy='MPC', horizon=300):
    env = boot.make_env(monitor=False)
    obs,_ = env.reset(seed=SEED+777)
    
    if policy=='MPC':
        # Planlayıcıyı en güncel modelle kullandığımızdan emin olalım
        planner_eval = RandomShootingMPC(model, ACT_LOW, ACT_HIGH, horizon=15, n_samples=4096, device=device, gauss_std=0.8)
        def act_fn(s): return planner_eval.act(s)
    else: # 'PPO'
        def act_fn(s):
            a,_ = ppo_baseline.predict(s, deterministic=True); return a
            
    real_obs, model_preds, actions = [obs.copy()], [obs.copy()], []
    
    # Gerçek ortamda bir yörünge topla
    for _ in range(horizon):
        a = act_fn(obs)
        obs, r, done, trunc, _ = env.step(a)
        real_obs.append(obs.copy())
        actions.append(a.copy())
        if done or trunc: break
    env.close()
    
    real_obs, actions = np.array(real_obs), np.array(actions)
    
    # Aynı aksiyon dizisini kullanarak modelden tahminler al
    s = torch.tensor(real_obs[0], device=device).unsqueeze(0)
    with torch.no_grad():
        for t in range(len(actions)):
            a_t = torch.tensor(actions[t], device=device).unsqueeze(0)
            s = model.step(s, a_t)
            model_preds.append(s.squeeze(0).cpu().numpy())
            
    return real_obs, np.array(model_preds)

# DAgger sonrası son eğitilmiş modeli yükle
final_fm = MLPDynamics(obs_dim, act_dim, hidden=(256, 256), layernorm=True).to(device)
final_fm.load_state_dict(torch.load(ckpt_fm, map_location=device))

real_mpc, pred_mpc = rollout_compare(final_fm, policy='MPC', horizon=300)
real_ppo, pred_ppo = rollout_compare(final_fm, policy='PPO', horizon=300)

def plot_series(real, pred, title):
    T=min(len(real), len(pred))
    fig,axes=plt.subplots(real.shape[1], 1, figsize=(12, 2 * real.shape[1]), sharex=True)
    for i,ax in enumerate(axes):
        ax.plot(real[:T,i], label='Gerçek Ortam', linewidth=2, alpha=0.8)
        ax.plot(pred[:T,i], label='Dinamik Model', linewidth=2, linestyle='--')
        ax.set_ylabel(labels[i]); ax.grid(True, linestyle=':')
        if i==0: ax.set_title(title, fontsize=14)
    axes[-1].set_xlabel('Adım'); axes[0].legend(); plt.tight_layout(); plt.show()

def rms(a): return float(np.sqrt(np.mean(a*a)))
def seq_mse(a,b): return float(np.mean((a-b)**2))

plot_series(real_mpc, pred_mpc, 'Dinamik Model vs Gerçek Ortam — MPC Politikası ile')
plot_series(real_ppo, pred_ppo, 'Dinamik Model vs Gerçek Ortam — PPO Politikası ile')

print(f"[Karşılaştırma/Metrikler] MPC: x_RMSE={rms(pred_mpc[:,0]-real_mpc[:,0]):.4f}, theta_RMSE={rms(pred_mpc[:,2]-real_mpc[:,2]):.4f}, Toplam MSE={seq_mse(pred_mpc, real_mpc):.4f}")
print(f"[Karşılaştırma/Metrikler] PPO: x_RMSE={rms(pred_ppo[:,0]-real_ppo[:,0]):.4f}, theta_RMSE={rms(pred_ppo[:,2]-real_ppo[:,2]):.4f}, Toplam MSE={seq_mse(pred_ppo, real_ppo):.4f}")

## 10) Return — MPC vs PPO (güvenli TB okuyucu ile)

In [ ]:
def tb_series(run_dir, tag):
    files=[]
    for root,_,fs in os.walk(run_dir):
        for f in fs:
            if f.startswith('events.out.tfevents'): files.append(os.path.join(root,f))
    if not files: return None
    dfs=[]
    for f in files:
        try:
            acc = ea.EventAccumulator(f, size_guidance={'scalars': 10**7}); acc.Reload()
            if tag in acc.Tags().get('scalars', []):
                s = acc.Scalars(tag)
                dfs.append(pd.DataFrame({'step':[i.step for i in s], 'val':[i.value for i in s]}))
        except Exception as e:
            print('[UYARI] TB okunamadı:', f, e)
    if not dfs: return None
    df = pd.concat(dfs).groupby('step', as_index=False)['val'].mean().sort_values('step')
    return df

def read_max_steps_from_csv(log_dir):
    p = os.path.join(log_dir, 'progress.csv')
    if not os.path.exists(p): return None
    try:
        df = pd.read_csv(p)
        for c in ['time/total_timesteps','timesteps','total_timesteps']:
            if c in df.columns: return int(np.nanmax(df[c].values))
    except: pass
    return None

plt.figure(figsize=(10, 6))

# PPO verisini çiz
ppo_df = tb_series(BASE_LOG, 'rollout/ep_rew_mean')
if ppo_df is not None:
    plt.plot(ppo_df['step'], ppo_df['val'], label='PPO (Baseline)', linewidth=2)
    
# DAgger boyunca MPC performansını çiz
dagger_df = tb_series(DAGGER_LOG, 'eval/return_mean')
if dagger_df is not None:
    # Adım sayısını DAgger iterasyonlarından toplam veri boyutuna çevir
    dagger_size_df = tb_series(DAGGER_LOG, 'sizes/obs')
    if dagger_size_df is not None:
        # Adımları veri boyutuyla eşleştir
        step_to_size = dict(zip(dagger_size_df['step'], dagger_size_df['val']))
        dagger_df['total_samples'] = dagger_df['step'].map(step_to_size)
        dagger_df.dropna(inplace=True)
        plt.plot(dagger_df['total_samples'], dagger_df['val'], label='MPC (DAgger ile eğitilmiş)', marker='o', linestyle='--')

plt.title('Performans Karşılaştırması: MPC vs PPO', fontsize=14)
plt.xlabel('Toplam Adım Sayısı (Timesteps)', fontsize=12)
plt.ylabel('Ortalama Bölüm Ödülü', fontsize=12)
plt.grid(True, linestyle=':', linewidth=0.8)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 11) Pygame Render — MPC & PPO

In [ ]:
import pygame

def _make_rgb_env(env_id=ENV_ID, seed=SEED):
    env = gym.make(env_id, render_mode='rgb_array'); obs, info = env.reset(seed=seed); return env, obs

def _pygame_init(width, height, title='RL Demo'):
    pygame.init(); surf = pygame.display.set_mode((width, height)); pygame.display.set_caption(title); clock = pygame.time.Clock(); return surf, clock

def _pygame_frame(surf, frame):
    frame = np.transpose(frame, (1,0,2))
    pg_surface = pygame.surfarray.make_surface(frame); surf.blit(pg_surface,(0,0)); pygame.display.flip()

def _handle_events():
    for event in pygame.event.get():
        if event.type == pygame.QUIT: return True
        if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE: return True
    return False

def render_demo(policy='MPC', episodes=2, max_steps=1000, fps=30):
    assert policy in ['MPC','PPO']
    env, obs = _make_rgb_env(); first = env.render(); H,W = first.shape[0], first.shape[1]
    surf, clock = _pygame_init(W, H, title=f'{policy} Demo — {ENV_ID}')
    
    # En son eğitilmiş modeli yükle
    final_fm_demo = MLPDynamics(obs_dim, act_dim, hidden=(256, 256), layernorm=True).to(device)
    final_fm_demo.load_state_dict(torch.load(ckpt_fm, map_location=device))

    if policy=='MPC':
        demo_planner = RandomShootingMPC(final_fm_demo, ACT_LOW, ACT_HIGH, horizon=15, n_samples=4096, device=device, gauss_std=0.8)
        def act_fn(s): return demo_planner.act(s)
    else:
        def act_fn(s):
            a,_ = ppo_baseline.predict(s, deterministic=True); return a
    try:
        for ep in range(episodes):
            obs,_ = env.reset(seed=SEED+300+ep); ret=0.0
            for t in range(max_steps):
                if _handle_events(): print('[INFO] Demo kapatıldı.'); return
                a = act_fn(obs); obs,r,done,trunc,_ = env.step(a); ret += r
                frame = env.render(); _pygame_frame(surf, frame); clock.tick(fps)
                if done or trunc: break
            print(f'[{policy} DEMO] Bölüm {ep+1}/{episodes} | Ödül={ret:.1f}')
    finally:
        pygame.quit(); env.close()

print("Render demolarını çalıştırmak için aşağıdaki satırların yorumunu kaldırın:")
# print("\n--- MPC Politikası Demosu ---")
# render_demo(policy='MPC', episodes=2, max_steps=1000, fps=30)
# print("\n--- PPO Politikası Demosu ---")
# render_demo(policy='PPO', episodes=2, max_steps=1000, fps=30)

## 12) Meta Bilgi (klasör/koşu özeti)

In [ ]:
meta = dict(phase=PHASE, notebook=NOTEBOOK, step=STEP, env=ENV_ID, seed=SEED,
            root_dir=boot.root_dir, run_id=boot.run_id,
            fm_log=FM_LOG, mpc_log=MPC_LOG, data_log=DATA_LOG, base_log=BASE_LOG, dagger_log=DAGGER_LOG,
            ckpt_fm=CKPT_FM_DIR, ckpt_mpc=CKPT_MPC_DIR, ckpt_base=CKPT_BASE_DIR, ckpt_dagger=CKPT_DAGGER_DIR)
with open(os.path.join(boot.root_dir, 'meta.json'), 'w') as f: json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))